In [1]:
import pandas as pd
import numpy as np
import os, sys
from tqdm.notebook import tqdm
from pathlib import Path
from plotly import express as px
import plotly.graph_objects as go
from sklearn.linear_model import  LinearRegression

In [2]:
file_path = Path("../rogii-wellbore-geology-prediction/train",)
well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }

In [3]:
well_files

{'000d7d20': {'Well': PosixPath('../rogii-wellbore-geology-prediction/train/000d7d20__horizontal_well.csv'),
  'TypeWell': PosixPath('../rogii-wellbore-geology-prediction/train/000d7d20__typewell.csv')},
 '00bbac68': {'Well': PosixPath('../rogii-wellbore-geology-prediction/train/00bbac68__horizontal_well.csv'),
  'TypeWell': PosixPath('../rogii-wellbore-geology-prediction/train/00bbac68__typewell.csv')},
 '00e12e8b': {'Well': PosixPath('../rogii-wellbore-geology-prediction/train/00e12e8b__horizontal_well.csv'),
  'TypeWell': PosixPath('../rogii-wellbore-geology-prediction/train/00e12e8b__typewell.csv')},
 '015fe0d2': {'Well': PosixPath('../rogii-wellbore-geology-prediction/train/015fe0d2__horizontal_well.csv'),
  'TypeWell': PosixPath('../rogii-wellbore-geology-prediction/train/015fe0d2__typewell.csv')},
 '01869cd4': {'Well': PosixPath('../rogii-wellbore-geology-prediction/train/01869cd4__horizontal_well.csv'),
  'TypeWell': PosixPath('../rogii-wellbore-geology-prediction/train/01869cd

In [4]:
# wells_data = pd.concat([
#     pd.read_csv(d["Well"]).loc[:,["X", "Y",'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',]] for d in tqdm(well_files.values())
#     ],axis=0).dropna()

In [5]:
# X = wells_data.loc[:, ["X", "Y"]]
# y = wells_data.loc[:,['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',]]

In [6]:
well_data = pd.read_csv(well_files["00bbac68"]["Well"])
pred_start = well_data.TVT_input.shift(1).dropna().index[-1]

In [7]:
formation_heights = well_data.loc[:,["X",'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']].melt(
    value_vars=['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'],ignore_index=False,
    var_name="Formation",value_name="Depth"
    )
formation_heights = pd.merge(well_data[["X","Y","Z","MD"]],formation_heights,left_index=True, right_index=True)


In [8]:
well_data

,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
0,11578.0,3010760.08,1084504.51,-9436.36,-9884.94,-10029.62,-10061.71,-10177.47,-10213.86,-10351.45,11406.63,99.123575,11406.63
1,11579.0,3010760.08,1084504.50,-9437.36,-9884.89,-10029.57,-10061.66,-10177.43,-10213.81,-10351.41,11407.67,100.422528,11407.67
2,11580.0,3010760.08,1084504.49,-9438.36,-9884.85,-10029.53,-10061.62,-10177.39,-10213.77,-10351.37,11408.71,102.602380,11408.71
3,11581.0,3010760.08,1084504.48,-9439.36,-9884.81,-10029.49,-10061.58,-10177.35,-10213.73,-10351.33,11409.75,NaN,11409.75
4,11582.0,3010760.08,1084504.47,-9440.36,-9884.78,-10029.46,-10061.55,-10177.32,-10213.70,-10351.30,11410.78,93.435055,11410.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7554,19132.0,3005792.08,1089518.64,-9996.34,-9634.87,-9779.55,-9811.64,-9927.41,-9963.79,-10101.39,12216.67,113.845045,NaN
7555,19133.0,3005791.35,1089519.32,-9996.34,-9634.84,-9779.52,-9811.61,-9927.38,-9963.76,-10101.36,12216.70,123.729035,NaN
7556,19134.0,3005790.61,1089520.00,-9996.34,-9634.81,-9779.49,-9811.58,-9927.35,-9963.73,-10101.32,12216.73,131.224145,NaN
7557,19135.0,3005789.88,1089520.68,-9996.34,-9634.78,-9779.46,-9811.55,-9927.31,-9963.70,-10101.29,12216.77,128.894987,NaN


In [9]:
well_data.loc[:,['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']].shape

(7559, 6)

In [12]:
pred_data = well_data.loc[pred_start:]
nearest_formation = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'][np.abs(pred_data.loc[:,['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']].values-pred_data[["Z"]].values).mean(axis=0).argmin()]

In [24]:
z_formation_deviance = well_data.loc[pred_start:,nearest_formation]-well_data.loc[pred_start:,"Z"]
px.line(z_formation_deviance - z_formation_deviance.iloc[0])

In [25]:
px.line(well_data.loc[pred_start:,"TVT"]-well_data.loc[pred_start,"TVT"])

In [ ]:
px.line(formation_heights,x="MD",y="Depth",color="Formation")